In [55]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
import pickle
import warnings
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree     import DecisionTreeClassifier
from sklearn.metrics  import (accuracy_score, f1_score, precision_score,
                               recall_score, confusion_matrix, classification_report)

In [61]:
df1=pd.read_csv("Reviews_cleaned")
df1

,Unnamed: 0,Sentiment
0,0,Positive
1,1,Neutral
2,2,Positive
3,3,Neutral
4,4,Positive
...,...,...
127909,127909,Neutral
127910,127910,Neutral
127911,127911,Positive
127912,127912,Positive


In [62]:
df2=pd.read_csv("Tf-Idf feature matrix")
df2

,Unnamed: 0,able,absolute,absolutely,accept,acceptable,acquired,actual,ad,add,...,yay,year,yellow,yogurt,yuck,yucky,yuk,yum,yummy,zero
0,0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,0.000000,0.0,0.0,0.0,0.147445,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,0.145234,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127909,127909,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
127910,127910,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
127911,127911,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
127912,127912,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [63]:
#Define X and y 
X = df2.values        # TF-IDF features (1000 word columns)
y = df1['Sentiment'] # Target labels (Positive / Negative / Neutral)
# Sanity check — both files must have same number of rows
if len(X) != len(y):
    print(f"ERROR: Row mismatch — Reviews has {len(y)} rows, "
          f"TF-IDF has {len(X)} rows.")

In [64]:
# Using only 20% of data (still ~25,000 rows — enough to train well) 
X_sample, _, y_sample, _ = train_test_split(
    X, y, train_size=0.2, random_state=42, stratify=y
)
print(f"Using {X_sample.shape[0]} rows for tuning (reduced from {X.shape[0]})")
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample, test_size=0.2, random_state=42, stratify=y_sample
)

Using 25582 rows for tuning (reduced from 127914)


In [65]:
# Cross-validation strategy
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
results = {}

In [45]:
# 1. LOGISTIC REGRESSION — Hyperparameter Tuning
lr_params = {
    'C':        [0.01, 0.1, 1, 10, 100],       # Regularization strength
    'penalty':  ['l1', 'l2'],                   # Type of regularization
    'solver':   ['liblinear', 'saga'],          # Compatible with l1 & l2
    'max_iter': [200, 500]
}

lr_grid = GridSearchCV(
    LogisticRegression(random_state=42),
    lr_params,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
lr_grid.fit(X_train, y_train)

lr_best   = lr_grid.best_estimator_
lr_pred   = lr_best.predict(X_test)

print("=" * 55)
print("LOGISTIC REGRESSION — Best Hyperparameters")
print("=" * 55)
print(lr_grid.best_params_)
print(f"Best CV Accuracy : {lr_grid.best_score_:.4f}")
print(f"Test Accuracy    : {accuracy_score(y_test, lr_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, lr_pred))

results['Logistic Regression'] = {
    'Best Params': lr_grid.best_params_,
    'CV Accuracy': lr_grid.best_score_,
    'Test Accuracy': accuracy_score(y_test, lr_pred)
}

Fitting 3 folds for each of 40 candidates, totalling 120 fits


C:\Users\TAKI-KUN\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
60 fits failed out of a total of 120.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
60 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\TAKI-KUN\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\TAKI-KUN\anaconda3\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\TAKI-KUN\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py", line 1208, in fit
 

LOGISTIC REGRESSION — Best Hyperparameters
{'C': 0.01, 'max_iter': 500, 'penalty': 'l2', 'solver': 'saga'}
Best CV Accuracy : 0.3336
Test Accuracy    : 0.3336

Classification Report:
              precision    recall  f1-score   support

    Negative       1.00      0.00      0.00      1705
     Neutral       0.33      1.00      0.50      1706
    Positive       0.00      0.00      0.00      1706

    accuracy                           0.33      5117
   macro avg       0.44      0.33      0.17      5117
weighted avg       0.44      0.33      0.17      5117



C:\Users\TAKI-KUN\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\TAKI-KUN\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\TAKI-KUN\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\TAKI-KUN\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in label

In [ ]:
labels = ['Negative', 'Neutral', 'Positive']

In [22]:
# 2. K-NEAREST NEIGHBORS — Hyperparameter Tuning
knn_params = {
    'n_neighbors': [3, 5, 7, 11, 15],          # Number of neighbors
    'weights':     ['uniform', 'distance'],     # Weight function
    'metric':      ['euclidean', 'manhattan'],  # Distance metric
    'p':           [1, 2]                       # Power for Minkowski metric
}

knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    knn_params,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
knn_grid.fit(X_train, y_train)

knn_best = knn_grid.best_estimator_
knn_pred = knn_best.predict(X_test)

print("=" * 55)
print("K-NEAREST NEIGHBORS — Best Hyperparameters")
print("=" * 55)
print(knn_grid.best_params_)
print(f"Best CV Accuracy : {knn_grid.best_score_:.4f}")
print(f"Test Accuracy    : {accuracy_score(y_test, knn_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, knn_pred))

results['KNN'] = {
    'best_params': knn_grid.best_params_,
    'cv_accuracy': knn_grid.best_score_,
    'test_accuracy': accuracy_score(y_test, knn_pred)
}

Fitting 3 folds for each of 40 candidates, totalling 120 fits


C:\Users\TAKI-KUN\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [0.33585778 0.34054795 0.33585778 0.34054795 0.33859365 0.33888658
 0.33859365 0.33888658 0.33742099 0.33766526 0.33742099 0.33766526
 0.33663959 0.33580861 0.33663959 0.33580861 0.34079236 0.3358576
 0.34079236 0.3358576         nan 0.33903334        nan 0.33903334
        nan 0.3401572         nan 0.3401572         nan 0.33786073
        nan 0.33786073        nan 0.33463611        nan 0.33463611
        nan 0.33517383        nan 0.33517383]
  warnings.warn(


K-NEAREST NEIGHBORS — Best Hyperparameters
{'metric': 'euclidean', 'n_neighbors': 15, 'p': 1, 'weights': 'uniform'}
Best CV Accuracy : 0.3408
Test Accuracy    : 0.3393

Classification Report:
              precision    recall  f1-score   support

    Negative       0.34      0.41      0.37      1705
     Neutral       0.33      0.31      0.32      1706
    Positive       0.34      0.29      0.31      1706

    accuracy                           0.34      5117
   macro avg       0.34      0.34      0.34      5117
weighted avg       0.34      0.34      0.34      5117



In [40]:
results['KNN'] = {
    'Best Params': knn_grid.best_params_,
    'CV Accuracy': knn_grid.best_score_,
    'Test Accuracy': accuracy_score(y_test, knn_pred)
}

In [35]:
results

{'SVM': {'CV Accuracy': 0.3334,
  'Test Accuracy': 0.3334,
  'Best Params': {'C': 0.1}},
 'Decision Tree': {'CV Accuracy': 0.5717,
  'Test Accuracy': 0.5845,
  'Best Params': {'criterion': 'gini',
   'max_depth': None,
   'min_samples_leaf': 1,
   'min_samples_split': 2}},
 'Random Forest': {'CV Accuracy': 0.7136,
  'Test Accuracy': 0.7149,
  'Best Params': {'criterion': 'gini',
   'max_depth': None,
   'min_samples_leaf': 1,
   'min_samples_split': 5,
   'n_estimators': 200}},
 'KNN': {'Best_Params': {'metric': 'euclidean',
   'n_neighbors': 15,
   'p': 1,
   'weights': 'uniform'},
  'CV_Accuracy': 0.3407923607986101,
  'Test_Accuracy': 0.3392612859097127}}

In [8]:
# 3. Linear SVC
from sklearn.svm import LinearSVC

svm_params = {
    'C': [0.1, 1, 10]               # Only tune C — kernel is fixed as linear
}

svm_grid = GridSearchCV(
    LinearSVC(random_state=42, max_iter=1000),
    svm_params, cv=cv, scoring='accuracy', n_jobs=-1, verbose=1
)
svm_grid.fit(X_train, y_train)
svm_pred = svm_grid.best_estimator_.predict(X_test)

print("\n" + "=" * 55)
print("SVM (LinearSVC) — Best Hyperparameters")
print("=" * 55)
print("Best Params  :", svm_grid.best_params_)
print(f"CV Accuracy  : {svm_grid.best_score_:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, svm_pred):.4f}")
print(classification_report(y_test, svm_pred))

results['SVM'] = {
    'CV Accuracy':   round(svm_grid.best_score_, 4),
    'Test Accuracy': round(accuracy_score(y_test, svm_pred), 4),
    'Best Params':   svm_grid.best_params_
}

Fitting 3 folds for each of 3 candidates, totalling 9 fits

SVM (LinearSVC) — Best Hyperparameters
Best Params  : {'C': 0.1}
CV Accuracy  : 0.3334
Test Accuracy: 0.3334
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00      1705
     Neutral       0.33      1.00      0.50      1706
    Positive       0.00      0.00      0.00      1706

    accuracy                           0.33      5117
   macro avg       0.11      0.33      0.17      5117
weighted avg       0.11      0.33      0.17      5117



C:\Users\TAKI-KUN\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\TAKI-KUN\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\TAKI-KUN\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [9]:
# 4. Decision Tree
dt_params = {
    'criterion':     ['gini', 'entropy'],
    'max_depth':     [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_params, cv=cv, scoring='accuracy', n_jobs=-1, verbose=1
)
dt_grid.fit(X_train, y_train)
dt_pred = dt_grid.best_estimator_.predict(X_test)

print("\n" + "=" * 55)
print("DECISION TREE — Best Hyperparameters")
print("=" * 55)
print("Best Params  :", dt_grid.best_params_)
print(f"CV Accuracy  : {dt_grid.best_score_:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, dt_pred):.4f}")
print(classification_report(y_test, dt_pred))

results['Decision Tree'] = {
    'CV Accuracy':   round(dt_grid.best_score_, 4),
    'Test Accuracy': round(accuracy_score(y_test, dt_pred), 4),
    'Best Params':   dt_grid.best_params_
}

Fitting 3 folds for each of 72 candidates, totalling 216 fits

DECISION TREE — Best Hyperparameters
Best Params  : {'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
CV Accuracy  : 0.5717
Test Accuracy: 0.5845
              precision    recall  f1-score   support

    Negative       0.59      0.58      0.58      1705
     Neutral       0.53      0.55      0.54      1706
    Positive       0.63      0.63      0.63      1706

    accuracy                           0.58      5117
   macro avg       0.59      0.58      0.58      5117
weighted avg       0.59      0.58      0.58      5117



In [68]:
# 5. RANDOM FOREST
rf_params = {
    'n_estimators': [100, 200],
    'max_depth':    [10, 20, None],
    'criterion':    ['gini', 'entropy'],
    'min_samples_split': [2, 5],
    'min_samples_leaf':  [1, 2]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_params, cv=cv, scoring='accuracy', verbose=1
)
rf_grid.fit(X_train, y_train)
rf_pred = rf_grid.best_estimator_.predict(X_test)

print("\n" + "=" * 55)
print("RANDOM FOREST — Best Hyperparameters")
print("=" * 55)
print("Best Params  :", rf_grid.best_params_)
print(f"CV Accuracy  : {rf_grid.best_score_:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, rf_pred):.4f}")
print(classification_report(y_test, rf_pred))

results['Random Forest'] = {
    'CV Accuracy':   round(rf_grid.best_score_, 4),
    'Test Accuracy': round(accuracy_score(y_test, rf_pred), 4),
    'Best Params':   rf_grid.best_params_
}

Fitting 3 folds for each of 48 candidates, totalling 144 fits

RANDOM FOREST — Best Hyperparameters
Best Params  : {'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
CV Accuracy  : 0.6984
Test Accuracy: 0.7209
              precision    recall  f1-score   support

    Negative       0.73      0.72      0.73      1706
     Neutral       0.67      0.65      0.66      1706
    Positive       0.76      0.79      0.78      1705

    accuracy                           0.72      5117
   macro avg       0.72      0.72      0.72      5117
weighted avg       0.72      0.72      0.72      5117



In [50]:
# 4. COMPARISON TABLE
comparison_df = pd.DataFrame(results).T
print("\n" + "=" * 55)
print("MODEL COMPARISON SUMMARY")
print("=" * 55)
comparison_df


MODEL COMPARISON SUMMARY


,CV Accuracy,Test Accuracy,Best Params
SVM,0.3334,0.3334,{'C': 0.1}
Decision Tree,0.5717,0.5845,"{'criterion': 'gini', 'max_depth': None, 'min_..."
Random Forest,0.7136,0.7149,"{'criterion': 'gini', 'max_depth': None, 'min_..."
KNN,0.340792,0.339261,"{'metric': 'euclidean', 'n_neighbors': 15, 'p'..."
Logistic Regression,0.333561,0.333594,"{'C': 0.01, 'max_iter': 500, 'penalty': 'l2', ..."


In [69]:
import pickle
# Save the best Random Forest model
with open('random_forest_model.pkl', 'wb') as f:
    pickle.dump(rf_grid.best_estimator_, f)

print(" Models saved successfully!")
print(f"   Best DT Params : {rf_grid.best_params_}")

 Models saved successfully!
   Best DT Params : {'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
   Best CV Acc    : 0.6984
